In [2]:
%%capture
%pip install mlflow
%pip install --upgrade torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu126

In [52]:
from collections import defaultdict
from datetime import datetime, timezone
import inspect
import json
import logging
import os
from pathlib import Path
import random
import shutil
import sys
import warnings
import zipfile

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_curve

import sqlite3

import torch
from torch import nn, optim
from torch.amp import GradScaler, autocast
from torch.cuda import Event
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from torch.optim.lr_scheduler import (
    CosineAnnealingWarmRestarts,
    CosineAnnealingLR,
    OneCycleLR,
    ReduceLROnPlateau,
)
from torch.utils.data import Dataset, DataLoader
from torchmetrics.classification import (
    BinaryAccuracy,
    BinaryPrecision,
    BinaryRecall,
    BinaryF1Score,
    AUROC,
    AveragePrecision,
    PrecisionRecallCurve,
)
from transformers import PreTrainedTokenizerFast
from tqdm.notebook import tqdm

import zipfile

warnings.filterwarnings("ignore", message=".*Pickle or CloudPickle.*")

In [4]:
print(torch.__version__)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_arch_list())

x = torch.randn(32, 32, device="cuda")
print((x @ x).sum().item())
torch.cuda.synchronize()

2.8.0+cu126
Tesla T4
['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']
4.6762542724609375


In [5]:
INPUT_ROOT = Path(
    "/kaggle/input/datasets/amirmohamadaskari/"
    "quora-question-pairs-wordpiece-training-assets"
)
WORK_ROOT = Path("/kaggle/working/qqp")
WORK_ROOT.mkdir(parents=True, exist_ok=True)


# Copy the existing MLflow database into writable storage
DB_NAME = "mlflow-quora-questions-pairs.db"

SOURCE_DB = INPUT_ROOT / DB_NAME
WORKING_DB = WORK_ROOT / DB_NAME

if not SOURCE_DB.is_file():
    raise FileNotFoundError(
        f"MLflow database not found in Kaggle dataset: {SOURCE_DB}"
    )

if not WORKING_DB.exists():
    shutil.copy2(SOURCE_DB, WORKING_DB)
    print(">>> Existing MLflow database imported successfully!")
else:
    print(">>> Using the existing working MLflow database.")

for folder in [
    "data/processed",
    "artifacts/tokenizers/wordpiece_uncased_30k",
]:
    shutil.copytree(
        INPUT_ROOT / folder,
        WORK_ROOT / folder,
        dirs_exist_ok=True,
    )

for filename in [
    "requirements.py",
    "02-LSTM-attention-train.ipynb",
]:
    shutil.copy2(
        INPUT_ROOT / filename,
        WORK_ROOT / filename,
    )

os.chdir(WORK_ROOT)

if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

print("Project directory:", Path.cwd())
print("Setup complete.")

>>> Existing MLflow database imported successfully!
Project directory: /kaggle/working/qqp
Setup complete.


In [6]:
%load_ext requirements

In [7]:
class SystemConfig:
    IS_DETERMINISTIC = False
    _PROJECT_ROOT = Path.cwd()  # repository root
    _DB_PATH = _PROJECT_ROOT / "mlflow-quora-questions-pairs.db"
    MLFLOW_TRACKING_URI = f"sqlite:///{_DB_PATH.as_posix()}"
    NEXT_LINE_COUNTER = 180
    SEED = 28
    USED_SCALER = False
    NEXT_LINE_COUNTER = 180
    @staticmethod
    def get_device():
        '''
        Detects the best available device for PyTorch.
        Priority: TPU -> GPU (CUDA) -> CPU
        '''
        # 1. Check for TPU (requires torch_xla)
        try:
            import torch_xla.core.xla_model as xm
            device = xm.xla_device()
            print(f">>> Using TPU: {device}")
        except ImportError:
            # 2. Check for GPU (CUDA)
            if torch.cuda.is_available():
                device = torch.device("cuda")
                print(f">>> Using GPU: {torch.cuda.get_device_name(0)}")
            # 3. Fallback to CPU
            else:
                device = torch.device("cpu")
                print(">>> Using CPU")
                
        return device
    DEVICE = get_device.__func__()

    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

>>> Using GPU: Tesla T4


In [8]:
class PathConfig:
    ROOT_DIR = Path().cwd()

    # -------------------------
    # Data
    # -------------------------
    DATA_DIR = ROOT_DIR / "data"
    RAW_DATA_DIR = DATA_DIR / "raw"
    PROCESSED_DATA_DIR = DATA_DIR / "processed"

    TRAIN_CSV_PATH = PROCESSED_DATA_DIR / "train_split.csv"
    VALID_CSV_PATH = PROCESSED_DATA_DIR / "valid_split.csv"

    # -------------------------
    # Artifacts
    # -------------------------
    ARTIFACT_DIR = ROOT_DIR / "artifacts"

    # Persistent shared tokenizer artifact
    TOKENIZER_DIR = (
        ARTIFACT_DIR
        / "tokenizers"
        / "wordpiece_uncased_30k"
    )

    # Run-specific LSTM-Attention artifacts
    MODEL_ARTIFACT_DIR = (
        ARTIFACT_DIR
        / "models"
        / "lstm_attention_wordpiece"
    )

    CHECKPOINT_DIR = MODEL_ARTIFACT_DIR / "checkpoint"
    CONFIG_PATH = MODEL_ARTIFACT_DIR / "configs.json"
    HISTORY_PATH = MODEL_ARTIFACT_DIR / "training_history.json"
    LABEL_MAPPING_PATH = MODEL_ARTIFACT_DIR / "label_mapping.json"

    # -------------------------
    # Project files
    # -------------------------
    REQ_TEXT = ROOT_DIR / "requirements.txt"
    REQ_SCRIPT = ROOT_DIR / "requirements.py"
    MODEL_SCRIPT = ROOT_DIR / "model_architecture.py"

    NOTEBOOK_PATH = ROOT_DIR / "02-LSTM-attention-train.ipynb"

    # -------------------------
    # MLflow
    # -------------------------
    MLFLOW_DIR = ROOT_DIR / "mlruns"

    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v
            for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)
            and not isinstance(v, (classmethod, staticmethod))
        }

    @classmethod
    def update_requirements(cls):
        from IPython import get_ipython

        ipython = get_ipython()
        if ipython:
            ipython.run_line_magic(
                "updatereqs",
                str(cls.NOTEBOOK_PATH)
            )

In [9]:
class TokenConfig:
    PAD_TOKEN = '[PAD]'
    UNK_TOKEN = '[UNK]'
    PAD_IDX = 0
    UNK_IDX = 1
    MAX_LENGTH = 64
    VOCAB_SIZE = 30000
    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

In [10]:
class LoaderConfig:
    BATCH_SIZE = 128
    NUM_WORKERS = 0
    IS_PIN_MEMORY = True
    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

In [11]:
class ModelConfig:
    MODEL_TYPE = "LSTM_attention"
    ATTENTION_TYPE = "MultiHead-Bahdanau"
    TOKENIZER_TYPE = "Wordpiece"
    # Embedding
    LAYER_NORM_EMB = False
    EMBEDDING_TYPE = "Trainable-from-scratch"
    EMB_DIM = 100
    EMB_DP = 0.2
    # Model
    LOSS = "BCE with Logits"
    NUM_HEADS = 4
    BIDIRECTIONAL = True
    DROPOUT = 0.35
    HIDDEN_DIM = 384
    LSTM_OUT = HIDDEN_DIM*(2 if BIDIRECTIONAL else 1)
    ATTENTION_DROPOUT = 0.0
    LAYER_NORM_LSTM = False
    LAYER_NORM_ATTENTION = False
    ATTENTION_PROJECTION = False
    if ATTENTION_PROJECTION:
        PROJECT_DIM = HIDDEN_DIM // 2
    ENC_DIM = PROJECT_DIM if ATTENTION_PROJECTION else LSTM_OUT
    if LOSS == "Contrastive Loss":
        MARGIN = 1.0
    elif LOSS == "BCE with Logits":
        LABEL_SMOOTHING = 0.05
        FC_DIMS = [1024, 256]
        FC_DP = 0.4
        SIAMESE_SIMILARITY_PARM = ["Encoded Q1", "Encoded Q2", "Multiplication Q1, Q2", "Abs Subtract Q1, Q2", "Cosine Similarity"]
        MULTIPLE_FC_PARAM = sum(1 for param in SIAMESE_SIMILARITY_PARM
                     if "Q1" in param or "Q2" in param)
        INPUT_FC_DIM = MULTIPLE_FC_PARAM * ENC_DIM
        if any("Cosine" in param for param in SIAMESE_SIMILARITY_PARM):
            INPUT_FC_DIM += 1
    MASK_FILL_NUM = -1e10
    NUM_LAYERS = 2
    SKIP_CONNECTION = False
    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

In [12]:
class TrainConfig:
    LOSS = ModelConfig.LOSS
    CLIP_NORM = 1.5
    EARLY_STOP_METRIC = "loss"
    CHECKPOINT_METRIC = "F1Score"
    if CHECKPOINT_METRIC == "loss":
        CHECKPOINT_MODE = "min"
    else:
        CHECKPOINT_MODE = "max"
    SCHEDULER_METRIC = "loss"
    EARLY_STOP_MIN_DELTA = 1e-4
    if EARLY_STOP_METRIC in ["F1Score", "Accuracy", "Precision", "Recall"]:
        EARLY_STOP_MODE = "max"
    elif EARLY_STOP_METRIC == "loss":
        EARLY_STOP_MODE = "min"
    EARLY_STOP_PATIENCE = 5
    EPOCHS = 50
    LEARNING_RATE = 3e-4
    METRICS_THRESHOLD = 0.5
    SCHEDULER_TYPE = "ReduceLROnPlateau"
    if SCHEDULER_TYPE == "ReduceLROnPlateau":
        SCHEDULER_FACTOR = 0.5
        SCHEDULER_MIN_LR = 1e-7
        SCHEDULER_PATIENCE = 2
        SCHEDULER_THRESHOLD = 0.01
        SCHEDULER_THRESHOLD_MODE = "rel"
        if SCHEDULER_METRIC in ["F1Score", "Accuracy", "Precision", "Recall"]:
            SCHEDULER_MODE = "max"
        elif SCHEDULER_METRIC == "loss":
            SCHEDULER_MODE = "min"
    elif SCHEDULER_TYPE == "CosineAnnealing":
        SCHEDULER_ETA_MIN = 1e-6
    elif SCHEDULER_TYPE == "CosineAnnealingWarmRestarts":
        SCHEDULER_T_0 = 5
        SCHEDULER_T_MUT = 2
        SCHEDULER_ETA_MIN = 1e-6
    elif SCHEDULER_TYPE == "OneCycleLR":
        SCHEDULER_PCT_START = 0.3
        SCHEDULER_DIV_FACTOR = 25
        SCHEDULER_FINAL_DIV_FACTOR = 1000

    WEIGHT_DECAY = 1e-3
    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

In [13]:
class PostProcessingConfig:
    INFERENCE_THRESHOLD = 0.5
    METRICS_THRESHOLD = 0.5
    @classmethod
    def to_dict(cls):
        return {
            k.lower(): v for k, v in cls.__dict__.items()
            if not k.startswith("_")
            and not inspect.isroutine(v)   # functions, methods
            and not isinstance(v, (classmethod, staticmethod))
        }

In [14]:
def seed_everything(seed, deterministic=False):
    '''
    Ensures absolute reproducibility.
    '''
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    if deterministic:
        # Only use these for the final "Gold" run to ensure exact results
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        print(">>> Using STRICT Deterministic mode (Slower).")
    else:
        # Benchmark=True allows cuDNN to find the fastest kernels for your hardware
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
        print(">>> Using PROTOTYPING mode (Faster).")
    print(f">>> For Reproducibility, Everything seeded with {seed}!")



In [15]:
def set_scaler(config=SystemConfig):
    if config.USED_SCALER:
        scaler = GradScaler(device=sys_cfg.DEVICE.type, enabled=(sys_cfg.DEVICE.type == 'cuda'))
        print(">>> Scaler used in training")
    else:
        scaler = None
        print(">>> Scaler is not Used")

    return scaler

In [16]:
def clean_artifact_directory(artifact_dir: Path):
    if artifact_dir.exists():
        shutil.rmtree(artifact_dir)
        print(f">>> Cleaned local artifact directory: {artifact_dir}")
    artifact_dir.mkdir(parents=True, exist_ok=True)

In [17]:
sys_cfg = SystemConfig()
path_cfg = PathConfig()
token_cfg = TokenConfig()
train_cfg = TrainConfig()
loader_cfg = LoaderConfig()
model_cfg = ModelConfig()
postprc_cfg = PostProcessingConfig()
print(">>> All Configs are set successfully!")
path_cfg.update_requirements()
scaler = set_scaler(sys_cfg)
seed_everything(sys_cfg.SEED, deterministic=sys_cfg.IS_DETERMINISTIC)
print(f">>> Training on: {sys_cfg.DEVICE} with seed = {sys_cfg.SEED}")

>>> All Configs are set successfully!
>>> Scanning: 02-LSTM-attention-train.ipynb
>>> Detected third-party imports: ['IPython', 'matplotlib', 'mlflow', 'numpy', 'pandas', 'sklearn', 'torch', 'torch_xla', 'torchmetrics', 'tqdm', 'transformers']
>>> Skipping 'torch_xla' (no matching package found)
>>> Added: torch, transformers, pandas, numpy, tqdm, mlflow, ipython, torchmetrics, matplotlib, scikit-learn
>>> Total packages in requirements.txt: 10
>>> Scaler is not Used
>>> Using PROTOTYPING mode (Faster).
>>> For Reproducibility, Everything seeded with 28!
>>> Training on: cuda with seed = 28


In [18]:
clean_artifact_directory(path_cfg.MODEL_ARTIFACT_DIR)

In [19]:
def get_serializable_configs(configs_dict):
    """Create a JSON-serializable version of configs"""
    serializable = {}
    for section, params in configs_dict.items():
        if section not in ["system", "path"]:
            serializable[section] = params
            continue
            
        serializable[section] = {}
        
        for key, value in params.items():
            if isinstance(value, torch.device):
                # Convert torch.device to string
                serializable[section][key] = str(value)
            elif isinstance(value, Path):
                # Convert Path to string
                serializable[section][key] = str(value)
            elif key == "device":
                # Skip or convert device
                serializable[section][key] = str(value) if value.type == "cuda" else "cpu"
            else:
                serializable[section][key] = value
    
    return serializable

In [20]:
def configs_dict(config_path):
    configs = {}
    configs_names = [
        SystemConfig, PathConfig, TokenConfig, LoaderConfig, TrainConfig, ModelConfig
    ]
    for config in configs_names:
        cfg_clean = f"{config.__name__.replace("Config", "").lower()}"
        configs[cfg_clean] = config.to_dict()

    serializable_configs = get_serializable_configs(configs)
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(serializable_configs, f, ensure_ascii=False, indent=2)
    print(">>> Configs Saved as JSON File!")
    return configs

configs = configs_dict(path_cfg.CONFIG_PATH)

>>> Configs Saved as JSON File!


In [21]:
train_df = pd.read_csv(path_cfg.TRAIN_CSV_PATH)
valid_df = pd.read_csv(path_cfg.VALID_CSV_PATH)

train_df.head()

,question1,question2,is_duplicate
0,What is maximum steering wheel torque in truck?,What is a steering wheel's torque?,0
1,How much does it cost to build a website in In...,How much does it cost to build an ecommerce st...,0
2,Does time stop ever?,Can time stop?,1
3,Did matter exist before the big bang?,What happened before the Big Bang happened?,1
4,If I have a master's in operations research (a...,I am an international masters student with lit...,0


In [22]:
print(f"Number of all pairs in research: {len(train_df) + len(valid_df)}")

Number of all pairs in research: 404287


In [23]:
print("Train missing values:")
display(train_df[["question1", "question2", "is_duplicate"]].isna().sum())

print("Validation missing values:")
display(valid_df[["question1", "question2", "is_duplicate"]].isna().sum())

Train missing values:


question1       0
question2       1
is_duplicate    0
dtype: int64

Validation missing values:


question1       0
question2       0
is_duplicate    0
dtype: int64

In [24]:
train_df["is_duplicate"].value_counts().sort_index()

is_duplicate
0    229521
1    134337
Name: count, dtype: int64

In [25]:
not_dupl = (train_df["is_duplicate"] == 0).sum()
is_dupl = (train_df["is_duplicate"] == 1).sum()
total = len(train_df)
print(
    f"Is duplicate in % : {is_dupl / total * 100}\n"
    f"not duplicate in % : {not_dupl / total * 100}"
)

Is duplicate in % : 36.920172155071484
not duplicate in % : 63.079827844928516


In [26]:
def preprocess_df(df):
    df = df.copy()

    df['question1'] = df['question1'].fillna('')
    df['question2'] = df['question2'].fillna('')

    before = len(df)
    df = df[
        (df["question1"].str.strip() != "") &
        (df["question2"].str.strip() != "")
    ].reset_index(drop=True)
    after = len(df)
    print(f">>> Preprocessing complete! Removed empty rows: {before - after}")

    return df

In [27]:
train_df_preprc = preprocess_df(train_df)
valid_df_preprc = preprocess_df(valid_df)

>>> Preprocessing complete! Removed empty rows: 1
>>> Preprocessing complete! Removed empty rows: 0


In [28]:
class QuoraTokenizer:
    def __init__(self, tokenizer_dir, config=token_cfg):
        self.config = config

        self.tokenizer = PreTrainedTokenizerFast.from_pretrained(
            tokenizer_dir,
            local_files_only=True
        )

        self.vocab_size = len(self.tokenizer)
        self.pad_idx = self.tokenizer.pad_token_id
        self.unk_idx = self.tokenizer.unk_token_id

        print(">>> WordPiece tokenizer loaded!")
        print(f">>> Vocabulary size: {self.vocab_size:,}")
        print(f">>> PAD token: {self.tokenizer.pad_token} ({self.pad_idx})")
        print(f">>> UNK token: {self.tokenizer.unk_token} ({self.unk_idx})")

    def encode(self, text):
        encoding = self.tokenizer(
            text,
            add_special_tokens=False,
            truncation=True,
            padding="max_length",
            max_length=self.config.MAX_LENGTH
        )

        return encoding["input_ids"]

    def decode(self, ids, remove_pad=True):
        if remove_pad:
            ids = [
                idx for idx in ids
                if idx != self.pad_idx
            ]

        return self.tokenizer.decode(
            ids,
            skip_special_tokens=True
        )

    def save_label_mapping(self, path):
        path = Path(path)

        label_mapping = {
            "0": "different",
            "1": "duplicated"
        }

        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                label_mapping,
                f,
                ensure_ascii=False,
                indent=2
            )

        print(">>> Label mapping saved!")

In [29]:
tokenizer = QuoraTokenizer(
    tokenizer_dir=path_cfg.TOKENIZER_DIR,
    config=token_cfg
)
print(
    ">>> First 10 tokens:",
    [
        (i, tokenizer.tokenizer.convert_ids_to_tokens(i))
        for i in range(10)
    ]
)

tokenizer.save_label_mapping(path_cfg.LABEL_MAPPING_PATH)

>>> WordPiece tokenizer loaded!
>>> Vocabulary size: 30,000
>>> PAD token: [PAD] (0)
>>> UNK token: [UNK] (1)
>>> First 10 tokens: [(0, '[PAD]'), (1, '[UNK]'), (2, '[CLS]'), (3, '[SEP]'), (4, '[MASK]'), (5, '!'), (6, '"'), (7, '#'), (8, '$'), (9, '%')]
>>> Label mapping saved!


In [30]:
class QuoraDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.q1 = df["question1"].values
        self.q2 = df["question2"].values
        self.label = df["is_duplicate"].values
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.q1)

    def pos_class_weight(self, device):
        pos_num = (self.label == 1).sum()
        neg_num = (self.label == 0).sum()

        pos_weight = torch.tensor(
            neg_num / pos_num,
            dtype=torch.float32,
            device=device
        )

        return pos_weight

    def __getitem__(self, idx):
        encoded_q1 = self.tokenizer.encode(self.q1[idx])
        encoded_q2 = self.tokenizer.encode(self.q2[idx])

        label = self.label[idx]

        return {
            "q1": torch.tensor(encoded_q1, dtype=torch.long),
            "q2": torch.tensor(encoded_q2, dtype=torch.long),
            "label": torch.tensor(label, dtype=torch.float32)
        }

In [31]:
train_dataset = QuoraDataset(train_df_preprc, tokenizer=tokenizer)
val_dataset = QuoraDataset(valid_df_preprc, tokenizer=tokenizer)

In [32]:
for i in range(3):
    item = val_dataset[i]
    print(f"\nSample {i}")
    print("Encoded Question 1:")
    print(item["q1"].tolist())
    print(f"\tDecoded text: {tokenizer.decode(item["q1"].tolist())}")
    print("Encoded Question 2:")
    print(item["q2"].tolist())
    print(f"\tDecoded Question 2: {tokenizer.decode(item["q2"].tolist())}")
    print("\t\tLabel:", item["label"])


Sample 0
Encoded Question 1:
[1481, 51, 2344, 1505, 9432, 43, 10394, 35, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
	Decoded text: can i give my therapist a hug?
Encoded Question 2:
[1481, 51, 1934, 1505, 9432, 1484, 43, 10394, 35, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
	Decoded Question 2: can i ask my therapist for a hug?
		Label: tensor(0.)

Sample 1
Encoded Question 1:
[1467, 1462, 51, 2432, 3403, 4248, 1456, 4816, 2317, 2684, 35, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
	Decoded text: how do i run jio 4g in 3g mobile set?
Encoded Question 2:
[1467, 1462, 51, 2432, 4248, 2068, 1456, 4816, 2317, 35, 0, 0, 0, 0, 0, 0, 0,

In [33]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=loader_cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=loader_cfg.NUM_WORKERS,
    pin_memory=loader_cfg.IS_PIN_MEMORY
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=loader_cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=loader_cfg.NUM_WORKERS,
    pin_memory=loader_cfg.IS_PIN_MEMORY
)

In [34]:
batch = next(iter(train_dataloader))

print(f"Batched Input IDs Size: {batch["q1"].shape}")
print(f"Batched Input IDs Size: {batch["q2"].shape}")
print(f"Batched Labels size: {batch["label"].shape}")

Batched Input IDs Size: torch.Size([128, 64])
Batched Input IDs Size: torch.Size([128, 64])
Batched Labels size: torch.Size([128])


In [35]:
class AttentionHead(nn.Module):
    def __init__(self, hidden_dim, proj_dim, mask_fill_num=model_cfg.MASK_FILL_NUM,
                 dropout=model_cfg.ATTENTION_DROPOUT):
        super().__init__()
        self.W = nn.Linear(hidden_dim, proj_dim)           # project to subspace
        self.V = nn.Linear(proj_dim, 1, bias=False)        # score
        self.mask_fill_num = mask_fill_num
        self.dropout = nn.Dropout(dropout)

    def forward(self, lstm_output, mask):
        proj = self.W(lstm_output)                         # [B, L, proj_dim]
        energy = torch.tanh(proj)
        scores = self.V(energy).squeeze(-1)                # [B, L]
        scores = scores.masked_fill(
            mask == 0, 
            torch.finfo(scores.dtype).min
        )
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)

        # Pool from the projected features (not original lstm_output)
        masked_proj = proj * mask.unsqueeze(-1)
        context = torch.bmm(weights.unsqueeze(1), masked_proj).squeeze(1)   # [B, proj_dim]
        return context

In [36]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads=model_cfg.NUM_HEADS):
        super().__init__()
        head_dim = hidden_dim // num_heads
        self.heads = nn.ModuleList([
            AttentionHead(hidden_dim, head_dim) for _ in range(num_heads)
        ])
        self.out_linear = nn.Linear(head_dim*num_heads, hidden_dim)

    def forward(self, hidden_state, mask):
        x = torch.cat([h(hidden_state, mask) for h in self.heads], dim=-1)
        x = self.out_linear(x)
        return x

In [37]:
class QuoraSiameseClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        pad_idx,
        config=model_cfg
    ):
        super().__init__()

        self.config = config
        self.pad_idx = pad_idx

        # Trainable embedding initialized from scratch
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=config.EMB_DIM,
            padding_idx=pad_idx
        )

        self.emb_norm = nn.LayerNorm(config.EMB_DIM)
        self.emb_dropout = nn.Dropout(config.EMB_DP)

        self.LSTM = nn.LSTM(
            input_size=config.EMB_DIM,
            hidden_size=config.HIDDEN_DIM,
            bidirectional=config.BIDIRECTIONAL,
            num_layers=config.NUM_LAYERS,
            dropout=config.DROPOUT if config.NUM_LAYERS > 1 else 0.0,
            batch_first=True
        )

        self.lstm_norm = nn.LayerNorm(config.LSTM_OUT)

        self.attention = MultiHeadAttention(
            config.LSTM_OUT
        )

        if config.ATTENTION_PROJECTION:
            self.proj = nn.Linear(
                config.LSTM_OUT,
                config.PROJECT_DIM
            )
        else:
            self.proj = nn.Identity()

        self.attn_norm = nn.LayerNorm(
            config.LSTM_OUT
        )

        self.fc_dims = self._build_fc_layers(
            input_dim=config.INPUT_FC_DIM,
            fc_dims=config.FC_DIMS,
            dropout=config.FC_DP
        )

    def _build_fc_layers(self, input_dim, fc_dims, dropout):
        layers = []

        for dim in fc_dims:
            layers += [
                nn.Linear(input_dim, dim),
                nn.GELU(),
                nn.Dropout(dropout)
            ]
            input_dim = dim

        layers.append(
            nn.Linear(input_dim, 1)
        )

        return nn.Sequential(*layers)

    def _create_mask(self, question):
        return (question != self.pad_idx).float()

    def _encode(self, question):
        # [B, L] -> [B, L, EMB_DIM]
        emb = self.embedding(question)

        if self.config.LAYER_NORM_EMB:
            emb = self.emb_norm(emb)

        emb = self.emb_dropout(emb)

        # Only PAD tokens are masked
        mask = self._create_mask(question)

        out, _ = self.LSTM(emb)

        if self.config.LAYER_NORM_LSTM:
            out = self.lstm_norm(out)

        ctx = self.attention(out, mask)

        if self.config.LAYER_NORM_ATTENTION:
            ctx = self.attn_norm(ctx)

        return ctx

    def forward(self, q1, q2):
        h1 = self._encode(q1)
        h2 = self._encode(q2)

        h1 = self.proj(h1)
        h2 = self.proj(h2)

        cosine_sim = F.cosine_similarity(h1, h2).unsqueeze(-1)
        feat = torch.cat(
            [h1, h2, torch.abs(h1 - h2), h1 * h2, cosine_sim], dim=1
        )

        logits = self.fc_dims(feat)

        return logits.squeeze(-1)

In [38]:
def export_model_from_notebook(
    notebook_path: str,
    output_path: Path,
    class_names=(
        "AttentionHead",
        "MultiHeadAttention",
        "QuoraSiameseClassifier",
        "ModelConfig",
    ),
):
    """Export cells containing the requested top-level model classes."""
    import ast

    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    target_classes = set(class_names)
    extracted_cells = []

    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)

        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue

        defined_classes = {
            node.name
            for node in tree.body
            if isinstance(node, ast.ClassDef)
        }

        matched_classes = defined_classes & target_classes

        if matched_classes:
            extracted_cells.append((source, matched_classes))

    if not extracted_cells:
        raise ValueError(
            "No matching class definitions found in the notebook."
        )

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(
            "import inspect\n"
            "import torch\n"
            "import torch.nn as nn\n"
            "import torch.nn.functional as F\n\n"
        )

        for source, matched_classes in extracted_cells:
            clean_source = "\n".join(
                line.rstrip() for line in source.splitlines()
            )
            f.write(clean_source)
            f.write("\n\n")

            if "ModelConfig" in matched_classes:
                f.write("model_cfg = ModelConfig()\n\n")

    print(
        f">>> Exported {len(extracted_cells)} cell(s) → {output_path}"
    )

In [39]:
export_model_from_notebook(path_cfg.NOTEBOOK_PATH, path_cfg.MODEL_SCRIPT)

>>> Exported 4 cell(s) → /kaggle/working/qqp/model_architecture.py


In [40]:
model = QuoraSiameseClassifier(
    vocab_size=tokenizer.vocab_size,
    config=model_cfg,
    pad_idx=tokenizer.pad_idx
).to(sys_cfg.DEVICE)
print(model)

QuoraSiameseClassifier(
  (embedding): Embedding(30000, 100, padding_idx=0)
  (emb_norm): LayerNorm((100,), eps=1e-05, elementwise_affine=True)
  (emb_dropout): Dropout(p=0.2, inplace=False)
  (LSTM): LSTM(100, 384, num_layers=2, batch_first=True, dropout=0.35, bidirectional=True)
  (lstm_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (attention): MultiHeadAttention(
    (heads): ModuleList(
      (0-3): 4 x AttentionHead(
        (W): Linear(in_features=768, out_features=192, bias=True)
        (V): Linear(in_features=192, out_features=1, bias=False)
        (dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (out_linear): Linear(in_features=768, out_features=768, bias=True)
  )
  (proj): Identity()
  (attn_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (fc_dims): Sequential(
    (0): Linear(in_features=3073, out_features=1024, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=1024, ou

In [41]:
num_params = sum(p.numel() for p in model.parameters())
print(f"Total Number of Parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Trainable Number of Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

Total Number of Parameters: 12633737
Trainable Number of Parameters: 12633737


In [42]:
class EarlyStopping:
    def __init__(self, patience=5, mode="min", min_delta=0.0):
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta

        self.should_stop = False
        self.best_score = None
        self.counter = 0

    def step(self, current_score):
        if self.best_score is None:
            self.best_score = current_score
            return True

        if self.mode == "min":
            improvement = self.best_score - current_score > self.min_delta
        else:
            improvement = current_score - self.best_score > self.min_delta

        if improvement:
            self.best_score = current_score
            self.counter = 0
            return True
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
            return False

In [43]:
class TrainingHistory:
    def __init__(self):
        self.history = defaultdict(list)

    def update(self, train_loss, val_loss, train_metrics, val_metrics, optimizer):
        self.history["train_loss"].append(train_loss)
        self.history["val_loss"].append(val_loss)
        for k_t, v_t in train_metrics.items():
            self.history[f"train_{k_t.lower()}"].append(v_t)
        for k_v, v_v in val_metrics.items():
            self.history[f"val_{k_v.lower()}"].append(v_v)
        self.history["lr"].append(optimizer.param_groups[0]["lr"])

    def save(self, path: str):
        """Save training history to a JSON file."""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.history, f, ensure_ascii=False, indent=2)

        print(f"Training History saved successfully at {path}")

    @classmethod
    def load(cls, path: str):
        "Load training history from a JSON file."
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"No file Found at {path}")
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        obj = cls()
        obj.history = data

        return obj

In [44]:
# Identify the existing experiment
EXPERIMENT_NAME = (
    f"Quora-Question-Pairs/{model_cfg.MODEL_TYPE}"
)

# The Kaggle artifact location
KAGGLE_ARTIFACT_ROOT = (
    path_cfg.MLFLOW_DIR
    / "Quora-Question-Pairs"
    / model_cfg.MODEL_TYPE
).as_uri()

# Save the original Windows location for later
PATH_MAP_FILE = WORK_ROOT / "mlflow-path-map.json"

with sqlite3.connect(sys_cfg._DB_PATH) as conn:

    experiment = conn.execute(
        """
        SELECT experiment_id, artifact_location
        FROM experiments
        WHERE name = ?
        """,
        (EXPERIMENT_NAME,)
    ).fetchone()

    if experiment is None:
        raise RuntimeError(
            f"Experiment not found in the imported database: "
            f"{EXPERIMENT_NAME}"
        )

    experiment_id, original_location = experiment

    if PATH_MAP_FILE.exists():
        path_map = json.loads(PATH_MAP_FILE.read_text())
        if (
            path_map["experiment_id"] != experiment_id
            or path_map["kaggle_root"] != KAGGLE_ARTIFACT_ROOT
            or original_location != KAGGLE_ARTIFACT_ROOT
        ):
            raise RuntimeError("Unexpected MLflow path state.")
        print(">>> MLflow artifact path already configured.")

    else:
        if original_location == KAGGLE_ARTIFACT_ROOT:
            raise RuntimeError(
                "Experiment already points to Kaggle, "
                "but its original Windows path is unknown."
            )

        path_map = {
            "experiment_id": experiment_id,
            "windows_root": original_location,
            "kaggle_root": KAGGLE_ARTIFACT_ROOT,
        }

        PATH_MAP_FILE.write_text(
            json.dumps(path_map, indent=2),
            encoding="utf-8"
        )

        conn.execute(
            """
            UPDATE experiments
            SET artifact_location = ?
            WHERE experiment_id = ?
            """,
            (KAGGLE_ARTIFACT_ROOT, experiment_id)
        )

        print(">>> MLflow artifact path switched to Kaggle!")

print("Original path:", path_map["windows_root"])
print("Current path:", KAGGLE_ARTIFACT_ROOT)

>>> MLflow artifact path switched to Kaggle!
Original path: file:///C:/Users/98922/Documents/python_scripts/AI/projects/quora-questions-pairs/mlruns/Quora-Question-Pairs/LSTM_attention
Current path: file:///kaggle/working/qqp/mlruns/Quora-Question-Pairs/LSTM_attention


In [45]:
class MLflowTracker:
    def __init__(self, project_name, run_type, config_dict, mlflow_dir, tracking_uri=None):
        model_type = config_dict["model"]["model_type"]
        attention_type = config_dict["model"]["attention_type"]
        tokenizer_type = config_dict["model"]["tokenizer_type"]
        embedding_type = config_dict["model"]["embedding_type"]
        self.experiment_name = f"{project_name}/{model_type}"
        self.attn_type = attention_type
        self.model_type = model_type
        self.tokenizer_type = tokenizer_type
        self.embedding_type = embedding_type
        
        self.base_run_name = (
            f"{model_type}-{attention_type}-{tokenizer_type}"
        )
        self.run_type = run_type
        self.config_dict = config_dict
        if tracking_uri:
            mlflow.set_tracking_uri(tracking_uri)
        else:
            mlflow.set_tracking_uri("http://localhost:5000")

        self.experiment = mlflow.get_experiment_by_name(self.experiment_name)
        artifact_dir = (mlflow_dir / project_name / model_type).as_uri()
        if self.experiment is None:
            experiment_id = mlflow.create_experiment(
                name=self.experiment_name,
                artifact_location=artifact_dir
            )
            self.experiment = mlflow.get_experiment(experiment_id)
            print(f">>> Created new experiment: {self.experiment_name}")
        else:
            print(f">>> Using existing experiment: {self.experiment_name}")

        mlflow.set_experiment(self.experiment_name)

        self.run_name = self._generated_versioned_run_name()
        print(f">>> Run Name: {self.run_name}")

    def _generated_versioned_run_name(self):
        base = self.base_run_name
        experiment = self.experiment
        if self.experiment is None:
            return f"{base}-v1"
    
        runs_df = mlflow.search_runs(
            experiment_ids=[experiment.experiment_id],
            filter_string="tags.status = 'completed'"
        )
        if runs_df.empty:
            return f"{base}-v1"
    
        prefix = f"{base}-v"
        mask = runs_df["tags.mlflow.runName"].str.startswith(prefix, na=False)
        matching = runs_df.loc[mask, "tags.mlflow.runName"]
    
        if matching.empty:
            return f"{base}-v1"
    
        versions = []
        for name in matching:
            try:
                version_str = name.split("-v")[-1]
                versions.append(int(version_str))
            except (ValueError, IndexError):
                continue
        next_version = max(versions) + 1 if versions else 1
        return f"{base}-v{next_version}"
            
        
    def start_run(self):
        self.run = mlflow.start_run(run_name=self.run_name)
        self.run_id = self.run.info.run_id
        return self.run

    def log_param(self, param_name, param):
        mlflow.log_param(param_name, param)
    def log_params(self, params):
        mlflow.log_params(params)
    def log_metric(self, name, value, epoch):
        mlflow.log_metric(name, value, step=epoch)
    def log_config_params(self):
        for key, value in self.config_dict.items():
            if key not in ["system", "path"]:
                mlflow.log_params(value) 
        
    def _log_losses(self, train_loss, val_loss, epoch):
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
    def _log_metrics(self, metrics_dict, epoch, prefix):
        for metric, value in metrics_dict.items():
            mlflow.log_metric(f"{prefix}_{metric.lower()}", value, step=epoch)
    def log_epoch(self, epoch, train_loss, val_loss, train_results, val_results, lr):
        self._log_losses(train_loss, val_loss, epoch)
        self._log_metrics(train_results, epoch, prefix="train")
        self._log_metrics(val_results, epoch, prefix="val")
        mlflow.log_metric("learning_rate", lr, step=epoch)
    def save_state_dict(self, state_dict, checkpoint_path):
        mlflow.pytorch.save_state_dict(
            state_dict,
            path=checkpoint_path
        )

    def load_state_dict(self, model, checkpoint_path):
        loaded_state_dict = mlflow.pytorch.load_state_dict(
            checkpoint_path
        )
        model.load_state_dict(loaded_state_dict)
        return model
    def log_best_model(self, model):
        mlflow_logger = logging.getLogger("mlflow.pytorch")
        original_level = mlflow_logger.level
        mlflow_logger.setLevel(logging.ERROR)

        mlflow.pytorch.log_model(
            model,
            name="best_models",
            registered_model_name=self.run_name,
            serialization_format="pickle",
        )

        mlflow_logger.setLevel(original_level)
    def log_artifact(self, artifact_path):
        mlflow.log_artifact(artifact_path)

    def log_artifact_folder(self, artifact_folder):
        mlflow.log_artifacts(artifact_folder)

    def build_run_summary(self, best_threshold, training_metrics, calibrated_metrics, total_time, avg_time_per_epoch):
        summary = {
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "run_type": self.run_type,
            "model_type": self.model_type,
            "attention_type": self.attn_type,
            "tokenizer_type": self.tokenizer_type,
            "embedding_type": self.embedding_type,
            "run_id": self.run_id,
            "best_metrics_in_training": training_metrics,
            "best_threshold": best_threshold,
            "best_calibrated_metrics": calibrated_metrics,
            "total_training_time_in_min": total_time,
            "average_training_time_per_epoch": avg_time_per_epoch,
            "params": {}
        }
        for k, v in self.config_dict.items():
            if k not in ["system", "path"]:
                summary["params"][k] = v

        return summary
    def log_summary(self, summary, artifact_name="run_sammary.json"):
        mlflow.log_dict(summary, artifact_name)

    def log_history(self, history, artifact_name="training_history.json"):
        mlflow.log_dict(history, artifact_name)

    def set_final_tags(self, best_score, best_calib_score, best_threshold, total_training_time):
        mlflow.set_tags({
            "status": "completed",
            "model_type": self.model_type,
            "attention_type": self.attn_type,
            "tokenizer_type": self.tokenizer_type,
            "embedding_type": self.embedding_type,
            "best_training_score": best_score,
            "best_calibrated_score": best_calib_score,
            "best_threshold": best_threshold,
            "total_training_time_in_min": total_training_time
        })

In [46]:
class Trainer:
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        device,
        history,
        mlflow_tracker,
        config=train_cfg,
        scaler=None
    ):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.history = history
        self.config = config
        self.criterion = criterion
        self.criterion_eval = nn.BCEWithLogitsLoss()
        self.optimizer = optimizer
        self.tracker = mlflow_tracker
        self.scaler = scaler
        
        self.early_stopper = EarlyStopping(
            patience=config.EARLY_STOP_PATIENCE,
            mode=config.EARLY_STOP_MODE,
            min_delta=config.EARLY_STOP_MIN_DELTA
        )
        self.scheduler = self._create_scheduler(config.SCHEDULER_TYPE)

        threshold = config.METRICS_THRESHOLD
        self.best_threshold = threshold
        metrics_cls = [BinaryAccuracy, BinaryPrecision, BinaryRecall, BinaryF1Score]
        self.train_metrics = {m.__name__.replace("Binary", ""): m(threshold).to(device) for m in metrics_cls}
        self.train_metrics["AUROC"] = AUROC(task="binary").to(device)
        self.train_metrics["AveragePrecision"] = AveragePrecision(task="binary").to(device)
        self.val_metrics = {m.__name__.replace("Binary", ""): m(threshold).to(device) for m in metrics_cls}
        self.val_metrics['AUROC'] = AUROC(task="binary").to(device)
        self.val_metrics['AveragePrecision'] = AveragePrecision(task="binary").to(device)
        self.pr_curve = PrecisionRecallCurve(task="binary").to(device)

        self.best_checkpoint_score = float('-inf') if config.CHECKPOINT_MODE == 'max' else float('inf')
        self.checkpoint_mode = config.CHECKPOINT_MODE

        self.train_start = Event(enable_timing=True)
        self.train_end = Event(enable_timing=True)
        self.epoch_start = Event(enable_timing=True)
        self.epoch_end = Event(enable_timing=True)
        self.epoch_durations = []

        self.current_epoch = 0
        
    def _create_scheduler(self, scheduler_type):
        if scheduler_type == "CosineAnnealing":
            return CosineAnnealingLR(
                optimizer=self.optimizer,
                T_max=self.config.EPOCHS,
                eta_min=self.config.SCHEDULER_ETA_MIN
            )
        elif scheduler_type == "ReduceLROnPlateau":
            return ReduceLROnPlateau(
                optimizer=self.optimizer,
                patience=self.config.SCHEDULER_PATIENCE,
                factor=self.config.SCHEDULER_FACTOR,
                mode=self.config.SCHEDULER_MODE,
                min_lr=self.config.SCHEDULER_MIN_LR,
                threshold=self.config.SCHEDULER_THRESHOLD,
                threshold_mode=self.config.SCHEDULER_THRESHOLD_MODE
            )
        elif scheduler_type == "OneCycleLR":
            return OneCycleLR(
                optimizer=self.optimizer,
                max_lr=self.config.LEARNING_RATE,
                epochs=self.config.EPOCHS,
                steps_per_epoch=len(self.train_loader),
                pct_start=self.config.SCHEDULER_PCT_START,        
                div_factor=self.config.SCHEDULER_DIV_FACTOR,        
                final_div_factor=self.config.SCHEDULER_FINAL_DIV_FACTOR 
            )
        elif scheduler_type == "CosineAnnealingWarmRestarts":
            return CosineAnnealingWarmRestarts(
                optimizer=self.optimizer,
                T_0=self.config.SCHEDULER_T_0,
                T_mult=self.config.SCHEDULER_T_MULT,
                eta_min=self.config.SCHEDULER_ETA_MIN
            )
        elif scheduler_type in (None, "none"):
            print("Warning: No scheduler")
            return None
        else:
            raise ValueError(f"Unknown scheduler: {scheduler_type}")

    def _check_scheduler(self, scheduler_value):
        if self.scheduler:
            if isinstance(self.scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                old_lr = self.optimizer.param_groups[0]["lr"]
                self.scheduler.step(scheduler_value)
                new_lr = self.optimizer.param_groups[0]["lr"]
                if new_lr < old_lr:
                    print(f">>> LR reduced: {old_lr:.6f} → {new_lr:.6f}")
            else:
                self.scheduler.step()
    
    def _reset_metrics(self, metrics_dict):
        for metric in metrics_dict.values():
            metric.reset()
    
    def _update_metrics(self, metrics_dict, probs, labels):
        for metric in metrics_dict.values():
            metric.update(probs, labels)
    def _update_threshold(self, metrics_dict):
        for metric in metrics_dict.values():
            if hasattr(metric, "threshold"):
                metric.threshold = self.best_threshold
    
    def _compute_metrics(self, metrics_dict):
        return {k:m.compute().item() for k, m in metrics_dict.items()}

    def _get_metric_value(self, metric_name, val_loss, val_results):
        if metric_name == "loss":
            return val_loss
        else:
            return val_results[metric_name]

    def _is_checkpoint_better(self, current_value: float) -> bool:
        if self.checkpoint_mode == 'max':
            return current_value > self.best_checkpoint_score
        else:
            return current_value < self.best_checkpoint_score

    def _backprop_with_scaler(self, q1, q2, labels):
        device_type = "cuda" if "cuda" in str(self.device) else "cpu"
        with autocast(enabled=self.scaler is not None, device_type=device_type):
            logits = self.model(q1, q2)
            loss = self.criterion(logits, labels)
        if self.scaler is not None:
            self.scaler.scale(loss).backward()

            if self.config.CLIP_NORM is not None and self.config.CLIP_NORM > 0:
                self.scaler.unscale_(self.optimizer)
                clip_grad_norm_(self.model.parameters(), self.config.CLIP_NORM)
            else:
                self.scaler.unscale_(self.optimizer)

            self.scaler.step(self.optimizer)
            self.scaler.update()
        else:
            loss.backward()
            if self.config.CLIP_NORM is not None and self.config.CLIP_NORM > 0:
                clip_grad_norm_(self.model.parameters(), self.config.CLIP_NORM)
                
            self.optimizer.step()

        return loss, logits
            
    def train_one_epoch(self):
        self.model.train()
        self._update_threshold(self.train_metrics)
        self._reset_metrics(self.train_metrics)
        total_loss = 0.0

        for batch in tqdm(self.train_loader, desc="Train", leave=True):
            q1 = batch["q1"].to(self.device)
            q2 = batch["q2"].to(self.device)
            labels = batch["label"].to(self.device).long()

            self.optimizer.zero_grad()
            
            loss, logits = self._backprop_with_scaler(q1, q2, labels)
            
            total_loss += (loss.item() * q1.size(0))

            probs = torch.sigmoid(logits)
            self._update_metrics(self.train_metrics, probs, labels)

        total_loss /= len(self.train_loader.dataset)
        results = self._compute_metrics(self.train_metrics)
        return total_loss, results

    @torch.no_grad()
    def evaluate(self, use_optimal_threshold=True):
        self.model.eval()
        total_loss = 0.0
        all_logits, all_labels = [], []
    
        for batch in tqdm(self.val_loader, desc="Validation", leave=True):
            q1 = batch["q1"].to(self.device)
            q2 = batch["q2"].to(self.device)
            labels = batch["label"].to(self.device).long()
    
            logits = self.model(q1, q2)
            loss = self.criterion_eval(logits, labels.float())
            total_loss += loss.item() * q1.size(0)
    
            all_logits.append(logits)
            all_labels.append(labels)
    
        total_loss /= len(self.val_loader.dataset)
    
        all_logits = torch.cat(all_logits)
        all_labels = torch.cat(all_labels)
        all_probs = torch.sigmoid(all_logits)
    
        # Find optimal threshold using self.pr_curve (if use_optimal_threshold)
        if use_optimal_threshold:
            self.pr_curve.reset()
            all_labels = all_labels.long()
            self.pr_curve.update(all_probs, all_labels)
            precision, recall, thresholds = self.pr_curve.compute()
            # Ensure thresholds is not empty
            if thresholds.numel() > 0:
                f1_scores = (
                    2 * precision[:-1] * recall[:-1]
                    / (precision[:-1] + recall[:-1] + 1e-8)
                )
                best_idx = torch.argmax(f1_scores)
                self.best_threshold = thresholds[best_idx].item()
            else:
                self.best_threshold = self.config.METRICS_THRESHOLD
        else:
            self.best_threshold = self.config.METRICS_THRESHOLD
    
        self._update_threshold(self.val_metrics)
    
        # Reset, update, compute using existing helpers
        self._reset_metrics(self.val_metrics)
        self._update_metrics(self.val_metrics, all_probs, all_labels)
        results = self._compute_metrics(self.val_metrics)
    
        return total_loss, results
    def log_one_epoch(self, train_loss, train_results, val_loss, val_results, best_thresh, lr):
        print(
            f"Training Results:\n\tLoss --> {train_loss:.4f}"
        )
        train_string = ""
        val_string = ""
        for k, v in train_results.items():
            train_result = f"{k} --> {v:.4f} | "
            train_string += train_result
        print(f"\t{train_string}")
        print(f"\tLearning Rate --> {lr:.4f}\n")
        print(
            f"Validation Results:\n\tLoss --> {val_loss:.4f}"
            )
        print(f"\tOptimal Best Threhsold --> {best_thresh}")
        for k, v in val_results.items():
            val_result = f"{k} --> {v:.4f} | "
            val_string += val_result
        print(f"\t{val_string}")

    @torch.no_grad()
    def find_optimal_threshold(self):
        self.model.eval()
        _, results = self.evaluate(use_optimal_threshold=True)
        best_threshold = self.best_threshold
    
        print(f">>> Optimal threshold: {best_threshold:.4f}")
        print(f">>> At that threshold --> F1: {results['F1Score']:.4f}, Precision: {results['Precision']:.4f}, Recall: {results['Recall']:.4f}")
        return best_threshold, results
    
    def fit(self, num_epochs, config=path_cfg):
        self.train_start.record()
        
        print(">>> Training Started...")
        with self.tracker.start_run() as run:
            self.tracker.log_config_params()
            
            self.tracker.log_artifact_folder(config.TOKENIZER_DIR)
            
            self.tracker.log_artifact(config.LABEL_MAPPING_PATH)
            self.tracker.log_artifact(config.CONFIG_PATH)
            self.tracker.log_artifact(config.MODEL_SCRIPT)
            self.tracker.log_artifact(config.REQ_SCRIPT)
            self.tracker.log_artifact(config.REQ_TEXT)

            best_metrics = {}
            
            for epoch in range(num_epochs):
                self.current_epoch = epoch
                print(f"Epoch {epoch+1}/{num_epochs}")
                self.epoch_start.record()
                torch.cuda.synchronize()
                
                train_loss, train_results = self.train_one_epoch()
                val_loss, val_results = self.evaluate()

                early_stop_value = self._get_metric_value(
                    self.config.EARLY_STOP_METRIC, val_loss, val_results
                )
                scheduler_value = self._get_metric_value(
                    self.config.SCHEDULER_METRIC, val_loss, val_results
                )
                checkpoint_value = self._get_metric_value(
                    self.config.CHECKPOINT_METRIC, val_loss, val_results
                )
        
                self._check_scheduler(scheduler_value)
                    
                self.history.update(
                    train_loss=train_loss,
                    val_loss=val_loss,
                    train_metrics=train_results,
                    val_metrics=val_results,
                    optimizer=self.optimizer
                )
                self.tracker.log_history(dict(self.history.history))
                lr = self.optimizer.param_groups[0]["lr"]
                self.log_one_epoch(
                    train_loss=train_loss,
                    train_results=train_results,
                    val_loss=val_loss,
                    val_results=val_results,
                    best_thresh=self.best_threshold,
                    lr=lr
                )
                self.tracker.log_epoch(
                    epoch=epoch,
                    train_loss=train_loss,
                    val_loss=val_loss,
                    train_results=train_results,
                    val_results=val_results,
                    lr=lr
                )
                self.tracker.log_metric("best_threshold", self.best_threshold, epoch)
                self.early_stopper.step(early_stop_value)
                is_better_checkpoint = False
                if self._is_checkpoint_better(checkpoint_value):
                    self.best_checkpoint_score = checkpoint_value 

                    best_metrics = val_results
                    self.tracker.save_state_dict(self.model.state_dict(), config.CHECKPOINT_DIR)
                    print(f">>> Best model saved! ({self.config.CHECKPOINT_METRIC} --> {checkpoint_value:.4f})")
                    
                if self.early_stopper.should_stop:
                    print(f">>> Early stopping triggered at Epoch {epoch+1}")
                    break

                self.epoch_end.record()
                torch.cuda.synchronize()
                
                epoch_duration = self.epoch_start.elapsed_time(self.epoch_end) / 1000
                self.epoch_durations.append(epoch_duration)
                print("="*sys_cfg.NEXT_LINE_COUNTER)


            self.train_end.record()
            torch.cuda.synchronize()
            total_training_time = self.train_start.elapsed_time(self.train_end) / 1000
            avg_time_per_epoch = sum(self.epoch_durations) / len(self.epoch_durations)
            total_training_time_in_min = round(total_training_time/60, 2)
            avg_time_per_epoch_in_min = round(avg_time_per_epoch/60, 2)

            self.model = self.tracker.load_state_dict(self.model, config.CHECKPOINT_DIR)
            self.model.eval()
            final_best_threshold, calibrated_results = self.find_optimal_threshold()
            self.tracker.log_artifact_folder(config.CHECKPOINT_DIR)
            self.tracker.log_best_model(self.model)
            print(">>> The Best Model registered at MLflow successfully!")
            summary = self.tracker.build_run_summary(
                best_threshold=final_best_threshold,
                training_metrics=best_metrics,
                calibrated_metrics=calibrated_results,
                total_time=total_training_time_in_min,
                avg_time_per_epoch=avg_time_per_epoch_in_min
            )
            self.tracker.log_param(
                f"best_training_{self.config.CHECKPOINT_METRIC}", 
                self.best_checkpoint_score
            )
            self.tracker.log_param(
                f"best_calibrated_{self.config.CHECKPOINT_METRIC}", 
                calibrated_results[self.config.CHECKPOINT_METRIC]
            )
            self.tracker.log_param("best_threshold", final_best_threshold)
            self.tracker.log_params(calibrated_results)
            self.tracker.log_param("total_training_time_in_min", total_training_time_in_min)
            self.tracker.log_param("average_time_per_epoch_in_min", avg_time_per_epoch_in_min)
            self.tracker.log_summary(summary)
            self.tracker.log_history(dict(self.history.history))
            self.tracker.log_artifact(config.NOTEBOOK_PATH)
            self.tracker.set_final_tags(
                best_score=self.best_checkpoint_score,
                best_calib_score=calibrated_results[self.config.CHECKPOINT_METRIC],
                best_threshold=final_best_threshold,
                total_training_time=total_training_time_in_min
            )

In [47]:
class BCEWithLabelSmoothing(nn.Module):
    def __init__(self,  epsilon, reduction="mean"):
        super().__init__()
        self.epsilon = epsilon
        self.criterion = nn.BCEWithLogitsLoss(reduction=reduction)

    def forward(self, logits, labels):
        labels = labels.float()
        labels = labels * (1 - self.epsilon) + (1 - labels) * self.epsilon
        return self.criterion(logits, labels)

In [48]:
mlflow_tracker = MLflowTracker(
    project_name="Quora-Question-Pairs",
    run_type="exploring-best-architecture",
    config_dict=configs,
    mlflow_dir=path_cfg.MLFLOW_DIR,
    tracking_uri=sys_cfg.MLFLOW_TRACKING_URI
)
history = TrainingHistory()
criterion = BCEWithLabelSmoothing(
    epsilon=model_cfg.LABEL_SMOOTHING
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_cfg.LEARNING_RATE,
    weight_decay=train_cfg.WEIGHT_DECAY
)

2026/09/21 10:47:15 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/21 10:47:15 INFO mlflow.store.db.utils: Updating database tables


>>> Using existing experiment: Quora-Question-Pairs/LSTM_attention
>>> Run Name: LSTM_attention-MultiHead-Bahdanau-Wordpiece-v1


In [49]:
trainer = Trainer(
    model=model,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    criterion=criterion,
    optimizer=optimizer,
    device=sys_cfg.DEVICE,
    history=history,
    config=train_cfg,
    mlflow_tracker=mlflow_tracker,
    scaler=scaler
)

In [50]:
trainer.fit(
    num_epochs=train_cfg.EPOCHS,
    config=path_cfg,
)

>>> Training Started...
Epoch 1/50


Train:   0%|          | 0/2843 [00:00<?, ?it/s]

Validation:   0%|          | 0/316 [00:00<?, ?it/s]

Training Results:
	Loss --> 0.5578
	Accuracy --> 0.7359 | Precision --> 0.6749 | Recall --> 0.5493 | F1Score --> 0.6057 | AUROC --> 0.7972 | AveragePrecision --> 0.6960 | 
	Learning Rate --> 0.0003

Validation Results:
	Loss --> 0.4670
	Optimal Best Threhsold --> 0.45821696519851685
	Accuracy --> 0.7708 | Precision --> 0.6551 | Recall --> 0.8008 | F1Score --> 0.7206 | AUROC --> 0.8563 | AveragePrecision --> 0.7746 | 
>>> Best model saved! (F1Score --> 0.7206)
Epoch 2/50


Train:   0%|          | 0/2843 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/316 [00:00<?, ?it/s]

>>> Optimal threshold: 0.6785
>>> At that threshold --> F1: 0.8208, Precision: 0.7831, Recall: 0.8623


2026/09/21 13:44:50 WARNING mlflow.utils.requirements_utils: Found torch version (2.8.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.8.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/09/21 13:45:05 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.23.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.23.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


>>> The Best Model registered at MLflow successfully!


Successfully registered model 'LSTM_attention-MultiHead-Bahdanau-Wordpiece-v1'.
Created version '1' of model 'LSTM_attention-MultiHead-Bahdanau-Wordpiece-v1'.


In [51]:
print("MLflow directory:", path_cfg.MLFLOW_DIR)
print("Experiment artifact URI:", mlflow_tracker.experiment.artifact_location)
print("Current run artifact URI:", mlflow.get_run(mlflow_tracker.run_id).info.artifact_uri)

MLflow directory: /kaggle/working/qqp/mlruns
Experiment artifact URI: file:///kaggle/working/qqp/mlruns/Quora-Question-Pairs/LSTM_attention
Current run artifact URI: file:///kaggle/working/qqp/mlruns/Quora-Question-Pairs/LSTM_attention/32d196f3656d4a059580837ababd38cb/artifacts


In [53]:
# Read the saved original and Kaggle paths
path_map = json.loads(
    PATH_MAP_FILE.read_text(encoding="utf-8")
)

windows_root = path_map["windows_root"]
kaggle_root = path_map["kaggle_root"]
experiment_id = path_map["experiment_id"]

# Only rewrite paths belonging to our Kaggle artifact root
def restore_uri(uri):
    if not isinstance(uri, str):
        return uri

    if uri == kaggle_root or uri.startswith(kaggle_root + "/"):
        return windows_root + uri[len(kaggle_root):]

    return uri


with sqlite3.connect(sys_cfg._DB_PATH) as conn:

    # Restore the experiment's original location
    current_root = conn.execute(
        """
        SELECT artifact_location
        FROM experiments
        WHERE experiment_id = ?
        """,
        (experiment_id,)
    ).fetchone()

    if current_root is None or current_root[0] not in (
        windows_root, kaggle_root
    ):
        raise RuntimeError("Unexpected experiment artifact location.")

    conn.execute(
        """
        UPDATE experiments
        SET artifact_location = ?
        WHERE experiment_id = ?
        """,
        (windows_root, experiment_id)
    )

    # Restore paths of newly generated MLflow objects
    fields = [
        ("runs", "artifact_uri"),
        ("logged_models", "artifact_location"),
        ("model_versions", "source"),
        ("model_versions", "storage_location"),
    ]

    for table, column in fields:

        columns = {
            row[1]
            for row in conn.execute(f"PRAGMA table_info({table})")
        }

        if column not in columns:
            continue

        rows = conn.execute(
            f"SELECT rowid, {column} FROM {table}"
        ).fetchall()

        changed = 0

        for row_id, uri in rows:
            restored = restore_uri(uri)

            if restored != uri:
                conn.execute(
                    f"UPDATE {table} SET {column} = ? "
                    f"WHERE rowid = ?",
                    (restored, row_id)
                )
                changed += 1

        print(f">>> {table}.{column}: {changed} paths restored")

print(">>> MLflow database paths restored to Windows!")


# Create a consistent SQLite backup for export
EXPORT_DB = WORK_ROOT / "mlflow-export.db"

with sqlite3.connect(sys_cfg._DB_PATH) as source:
    with sqlite3.connect(EXPORT_DB) as destination:
        source.backup(destination)


# Package everything generated in this Kaggle session
EXPORT_ZIP = Path("/kaggle/working/qqp-transfer.zip")

with zipfile.ZipFile(
    EXPORT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=4
) as archive:

    # Updated database
    archive.write(
        EXPORT_DB,
        arcname=DB_NAME
    )

    # MLflow artifacts, including registered model files
    # No old local mlruns were uploaded, so they are not here.
    for file in path_cfg.MLFLOW_DIR.rglob("*"):
        if file.is_file():
            archive.write(
                file,
                arcname=file.relative_to(WORK_ROOT).as_posix()
            )

    # Your normal model artifacts/checkpoint
    for file in path_cfg.MODEL_ARTIFACT_DIR.rglob("*"):
        if file.is_file():
            archive.write(
                file,
                arcname=file.relative_to(WORK_ROOT).as_posix()
            )

print(f">>> Export ready: {EXPORT_ZIP}")

>>> runs.artifact_uri: 1 paths restored
>>> logged_models.artifact_location: 1 paths restored
>>> model_versions.source: 0 paths restored
>>> model_versions.storage_location: 1 paths restored
>>> MLflow database paths restored to Windows!
>>> Export ready: /kaggle/working/qqp-transfer.zip
